## Create the Catalog, schema, and volume

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.gh_archive;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main.gh_archive.bronze_raw;
CREATE VOLUME IF NOT EXISTS main.gh_archive.checkpoint_path;

In [0]:
import urllib.request
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import current_timestamp, col
import os
import re

# ---------------------------------------------------------------------------
# 0. Config
# ---------------------------------------------------------------------------
volume_path = "/Volumes/main/gh_archive/bronze_raw/"
checkpoint_path = "/Volumes/main/gh_archive/_checkpoints/bronze_events"
headers = {"User-Agent": "Mozilla/5.0"}
MAX_WORKERS = 16

# ---------------------------------------------------------------------------
# 1. Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("start_date", "", "Start date (YYYY-MM-DD, blank = auto)")
dbutils.widgets.text("end_date", "", "End date (YYYY-MM-DD, blank = today)")
dbutils.widgets.text("lookback_days", "7", "Fallback lookback if no prior data (days)")

start_date_param = dbutils.widgets.get("start_date").strip()
end_date_param = dbutils.widgets.get("end_date").strip()
lookback_days = int(dbutils.widgets.get("lookback_days"))

now_utc = datetime.now(timezone.utc)
latest_available_hour = (now_utc - timedelta(hours=2)).replace(minute=0, second=0, microsecond=0)

# ---------------------------------------------------------------------------
# 2. Resolve the date range
# ---------------------------------------------------------------------------
def get_latest_downloaded_hour(volume_path):
    pattern = re.compile(r"(\d{4}-\d{2}-\d{2})-(\d{1,2})\.json\.gz$")
    latest = None
    try:
        for fname in os.listdir(volume_path):
            m = pattern.search(fname)
            if m:
                date_str, hour = m.group(1), int(m.group(2))
                dt = datetime.strptime(date_str, "%Y-%m-%d").replace(hour=hour, tzinfo=timezone.utc)
                if latest is None or dt > latest:
                    latest = dt
    except FileNotFoundError:
        pass
    return latest

if start_date_param:
    range_start = datetime.strptime(start_date_param, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    print(f"Using explicit start_date param: {range_start.date()}")
else:
    latest_dt = get_latest_downloaded_hour(volume_path)
    if latest_dt:
        range_start = (latest_dt + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
        print(f"Incremental mode: resuming from {range_start} (latest existing file: {latest_dt})")
    else:
        range_start = (now_utc - timedelta(days=lookback_days)).replace(minute=0, second=0, microsecond=0)
        print(f"No existing data found. Falling back to lookback of {lookback_days} days: {range_start}")

if end_date_param:
    range_end = datetime.strptime(end_date_param, "%Y-%m-%d").replace(hour=23, tzinfo=timezone.utc)
else:
    range_end = latest_available_hour

if range_start > range_end:
    print(f"Nothing to do — range_start ({range_start}) is after range_end ({range_end}). Exiting.")
    dbutils.notebook.exit("no_new_data")

# ---------------------------------------------------------------------------
# 3. Build the job list — skip files already on disk (download idempotency)
# ---------------------------------------------------------------------------
jobs = []
cursor = range_start
while cursor <= range_end:
    date_str = cursor.strftime("%Y-%m-%d")
    hour = cursor.hour
    dest = f"{volume_path}{date_str}-{hour}.json.gz"
    if not os.path.exists(dest):
        jobs.append((date_str, hour))
    cursor += timedelta(hours=1)

print(f"Range: {range_start} -> {range_end}")
print(f"{len(jobs)} file(s) to download.")

# ---------------------------------------------------------------------------
# 4. Download in parallel
# ---------------------------------------------------------------------------
def download_one(date_str, hour):
    url = f"https://data.gharchive.org/{date_str}-{hour}.json.gz"
    dest = f"{volume_path}{date_str}-{hour}.json.gz"
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=30) as response, open(dest, "wb") as out_file:
            out_file.write(response.read())
        return (date_str, hour, True, None)
    except Exception as e:
        return (date_str, hour, False, str(e))

failed = []
if jobs:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(download_one, d, h) for d, h in jobs]
        for future in as_completed(futures):
            date_str, hour, ok, err = future.result()
            if ok:
                print(f"Downloaded {date_str}-{hour}")
            else:
                print(f"Failed {date_str}-{hour}: {err}")
                failed.append((date_str, hour, err))

print(f"Download done. {len(jobs) - len(failed)}/{len(jobs)} succeeded.")
if failed:
    print(f"{len(failed)} failure(s): {failed}")

# ---------------------------------------------------------------------------
# 5. Ingestion Phase — Auto Loader, checkpoint-tracked, idempotent
# ---------------------------------------------------------------------------
print("Starting incremental Bronze ingestion...")

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", checkpoint_path)
      .option("cloudFiles.schemaEvolutionMode", "rescue")
      .load(volume_path))

df = df.withColumn("_ingested_at", current_timestamp()) \
       .withColumn("_source_file", col("_metadata.file_path"))

query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .toTable("main.gh_archive.bronze_events"))

query.awaitTermination()

print("Bronze ingestion complete.")

In [0]:
%sql
SELECT COUNT(*)
FROM main.gh_archive.bronze_events